<div style="background:linear-gradient(120deg,#EDE7FF 0%,#E3F0FF 55%,#E6FBF7 100%); color:#2A1B5C; padding:22px 26px; border-radius:14px; border:1px solid #DCD3FF; margin-bottom:8px;">

<h1 style="margin:0 0 6px 0; color:#3B1F7A">09a · Generate synthetic visit notes (Google Colab, free GPU)</h1>

<p style="margin:0 0 10px 0; font-size:1.1em; color:#4C3A8C">Run this on Colab: an open 7-billion-parameter model writes ~1,200 rep notes, a quality gate keeps only faithful ones</p>

| | |
|---|---|
| **Runs on** | Google Colab free **T4 GPU** (not on the laptop) · ~1 hour |
| **Model** | `Qwen/Qwen2.5-7B-Instruct` (Apache 2.0), loaded in **4-bit** so it fits in 16 GB GPU memory |
| **Inputs** | scenarios + prompt + quality gate, downloaded from this project's GitHub repo |
| **Output** | `notes_raw.jsonl`: download it and put it in `data/04_synthetic_visit_notes/` |

</div>

<div style="border-left:6px solid #14B8A6; background:#E9FBF8; color:#134E4A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">🛠️ What to do</b>

**How to run (4 clicks):**
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**
3. Wait ~1 hour. You can leave the tab open in the background; progress prints after every batch.
4. The last cell downloads `notes_raw.jsonl`. Move it into the project folder `data/04_synthetic_visit_notes/`.

If Colab disconnects, just **Run all** again: finished notes are skipped (*resume*).

</div>

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 1 · Check the GPU

</div>

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU, then Run all again"
print("GPU:", torch.cuda.get_device_name(0), f"· {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 2 · Install and download project files

</div>

<div style="border-left:6px solid #FB923C; background:#FFF4E8; color:#7C2D12; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">💡 Concept · 4-bit quantisation</b>

**Quantisation** stores each model weight in 4 bits instead of 16, cutting memory ~4× (7B parameters: ~15 GB → ~5 GB) with only a small quality loss.
`bitsandbytes` does this on the fly when the model loads (*NF4* format). This is the same idea notebook 16 uses for QLoRA fine-tuning.

</div>

In [ ]:
!pip -q install -U bitsandbytes accelerate transformers

RAW = "https://raw.githubusercontent.com/Swagata-Bhowmik/GLP1-Launch-Intelligence/main"
!wget -q -O gate.py          {RAW}/src/glp1_intel/synthetic/gate.py
!wget -q -O open_generate.py {RAW}/src/glp1_intel/synthetic/open_generate.py
!wget -q -O scenarios.jsonl  {RAW}/data/04_synthetic_visit_notes/scenarios_v1.jsonl
!wget -q -O prompt.json      {RAW}/data/04_synthetic_visit_notes/prompt_v1.json
!ls -la gate.py open_generate.py scenarios.jsonl prompt.json

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 3 · Load the model in 4-bit

</div>

In [ ]:
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

prompt = json.load(open("prompt.json"))
scenarios = [json.loads(line) for line in open("scenarios.jsonl", encoding="utf-8")]
print(f"{len(scenarios):,} scenarios · model: {prompt['model']}")

quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(prompt["model"])
model = AutoModelForCausalLM.from_pretrained(prompt["model"], quantization_config=quant, device_map="auto")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 4 · Generate with the quality gate

</div>

<div style="border-left:6px solid #FB923C; background:#FFF4E8; color:#7C2D12; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">💡 Concept · Few-shot prompting + rejection sampling</b>

- The prompt has a **system message** (role and rules) and **two worked examples** (few-shot prompting), then the scenario's brief.
- **Sampling** (temperature 0.8, top-p 0.9) gives varied notes; 16 notes are generated **in parallel** per batch.
- Every note goes through `gate.check()`. A note that misses a product or competitor, mishandles the side effect, leaks prompt words or names a doctor is **regenerated** (up to 3 attempts).

</div>

In [ ]:
import open_generate

summary = open_generate.run(model, tok, scenarios, prompt, "notes_raw.jsonl",
                            batch_size=16, max_attempts=3, max_new_tokens=260)
summary

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 5 · Look at a few notes, then download

</div>

In [ ]:
import random
recs = [json.loads(l) for l in open("notes_raw.jsonl", encoding="utf-8")]
print(f"passed {sum(r['passed'] for r in recs):,} of {len(recs):,}")
for r in random.sample([r for r in recs if r["passed"]], 3):
    print(f"\n--- {r['scenario_id']} · {r['specialty']} · interest {r['interest_level']} · AE {r['ae_mode']} {r['ae_term']}\n{r['note']}")

In [ ]:
from google.colab import files
files.download("notes_raw.jsonl")      # → move into data/04_synthetic_visit_notes/ in the project